# Model evaluation: how well will predictions generalize?

## Introduction

A model can fit historical applications well and still disappoint on new borrowers. In this notebook, you revisit **Machine Learning Model Evaluation** through the synthetic credit-risk application. The worked examples show how data organization, classification metrics and model flexibility affect our assessment of predictions on unseen observations.

By the end, you can:

- use stratified cross-validation with explicitly named metrics;
- read a confusion matrix and calculate accuracy, recall, specificity and precision;
- plot an ROC curve from probabilities and interpret ROC-AUC;
- compare training and validation results before evaluating a chosen model on a held-out test set.

**Outline:** [Key Concepts](#key-concepts) · [Assessing prediction model performance](#assessing-prediction-model-performance) · [Bias, variance and overfitting](#bias-variance-and-overfitting) · [Summary](#summary) · [Tasks](#tasks)

**Lecture connection:** *Machine Learning Model Evaluation*, slides 10–23 (evaluation principles and classification metrics) and 26–32 (overfitting, bias and variance). Slides 6–7 revisit cross-validation.

We'll use **pandas** to organize predictions and results, and **Matplotlib** to visualize them. **Scikit-learn** supplies the models, validation procedures and performance metrics. The imports are collected below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
)

## Key Concepts

Evaluation needs two ingredients: **a performance measure** and **data the fitted model has not used for training**. A metric computed on the training sample describes fit; it does not directly estimate performance on new applications.

For borrower $i$, $y_i\in\{0,1\}$ is the observed outcome and $\widehat{p}_i$ is the estimated default probability. Throughout this notebook, **1 means default**. There are $n$ observations and $d=2$ application-time features. At a fixed threshold $c$, the class decision is

$$\widehat{y}_i(c)=\mathbb{I}\{\widehat{p}_i\geq c\}.$$

The indicator is one when the condition holds. We fix $c=0.5$ for the class-based calculations below. Choosing a threshold from test results would make that sample part of model selection rather than an independent assessment.

The confusion counts are **TP** (correctly flagged defaults), **TN** (correctly classified non-defaults), **FP** (non-defaults flagged as defaults), and **FN** (missed defaults). Common metrics are

$$\mathrm{Accuracy}=\frac{\mathrm{TP}+\mathrm{TN}}{n},\qquad
\mathrm{Recall}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}},$$
$$\mathrm{Specificity}=\frac{\mathrm{TN}}{\mathrm{TN}+\mathrm{FP}},\qquad
\mathrm{Precision}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}}.$$

Recall is also called sensitivity or true-positive rate. The false-positive rate is $1-\mathrm{Specificity}$. A ratio is undefined when its denominator is zero; software may replace that undefined value by a specified convention.

An **ROC curve** plots true-positive rate against false-positive rate across thresholds. Its area, **ROC-AUC**, summarizes ranking: a randomly selected default should receive a higher score than a randomly selected non-default (ties receive half credit). Unlike a confusion matrix, it does not summarize one fixed threshold.

**Bias** concerns systematic approximation error; **variance** concerns sensitivity to the training sample. A flexible model may fit both structure and sample-specific noise. Training–validation differences can signal overfitting, but one difference does not measure bias and variance separately.

## Assessing prediction model performance

### Load the credit applications

The demonstration uses the supplied local dataset from the credit-risk example. `bureau_score` is measured in score points on a stylized 300–850 scale. `debt_income` is monthly debt payments divided by gross monthly income, expressed in **percent** (35 means 35%). Both features are known at application; `default` records the following 12 months.

The borrowers are synthetic. This is a manageable example for practicing evaluation, rather than evidence for an operational lending policy.

In [ ]:
credit = pd.read_csv('../data/credit_data.csv')
feature_names = ['bureau_score', 'debt_income']
X = credit[feature_names]
y = credit['default']
credit.head()

### Reserve the test set before modeling

The next cell sets aside 25% of applications for a final assessment. Model comparisons use only the remaining **development data**. A stratified split approximately preserves the class proportions. The seed makes it reproducible; it does not make any one split definitive.

Here the synthetic applications are independent draws from the same setting, so a random split is appropriate. In practice, the evaluation population must represent the intended use of the model.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=67
)
print(f'Development applications: {len(X_dev)}; reserved test applications: {len(X_test)}')
print(f'Development default share: {y_dev.mean():.1%}')

### Cross-validation with explicit scoring

`StratifiedKFold` partitions the development sample into five folds with similar class proportions. Each fitted model uses four folds for training and predicts the remaining fold. Each borrower appears in a validation fold exactly once.

`cross_validate` creates a fresh fitted model for each fold. `scoring` states precisely what we measure: **accuracy** uses class labels, whereas **ROC-AUC** uses continuous prediction scores. Specifying the metrics avoids relying on an estimator's default `.score` behavior. `return_train_score=True` also reports fit on each fold's training observations.

The keys beginning with `test_` in the returned dictionary refer to **validation folds**, not our reserved final test set. Cross-validation trains on fewer cases than the final fitted model, so it estimates the performance of that training procedure at the fold training size.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
logit = LogisticRegression(solver='newton-cholesky')
cv_results = cross_validate(
    logit, X_dev, y_dev, cv=folds,
    scoring={'accuracy': 'accuracy', 'roc_auc': 'roc_auc'},
    return_train_score=True
)
fold_results = pd.DataFrame({
    'training_accuracy': cv_results['train_accuracy'],
    'validation_accuracy': cv_results['test_accuracy'],
    'validation_roc_auc': cv_results['test_roc_auc']
}, index=pd.Index(range(1, 6), name='fold'))
fold_results.round(3)

In [ ]:
fold_summary = pd.DataFrame({
    'mean': fold_results.mean(),
    'standard_deviation': fold_results.std()
})
fold_summary.round(3)

Inspect both the means and the fold-to-fold variation. A small training–validation gap is reassuring, but it does not prove that future performance will be identical. These fold results share overlapping training samples: their standard deviation is a descriptive measure of variation, not a confidence interval.

### Build a table of out-of-fold predictions

`cross_val_predict` returns one prediction for each development row, in the original row order. Each prediction comes from a model that did **not** train on that row. With `method='predict_proba'`, it returns class probabilities; for our `[0, 1]` target, column 1 is default probability.

This call fits the five models again to collect their row-level predictions. It does not fit a single final model. The resulting table lets you connect probabilities, decisions and observed outcomes.

In [ ]:
oof_probability = cross_val_predict(
    logit, X_dev, y_dev, cv=folds, method='predict_proba'
)[:, 1]
cutoff = 0.5
oof_class = (oof_probability >= cutoff).astype(int)
predictions = X_dev.copy()
predictions['observed_default'] = y_dev
predictions['P(default)'] = oof_probability
predictions['predicted_default'] = oof_class
predictions.head().round(3)

### Confusion matrix: start with the axis labels

![Evaluation lecture slide 17: confusion counts and formulas for accuracy, specificity, recall and precision.](../resources/3_classification_metrics.png)

The lecture places **predicted** classes in rows and **actual** classes in columns. Scikit-learn uses the **transpose**: rows are actual classes and columns are predicted classes. With `labels=[0, 1]`, the array is `[[TN, FP], [FN, TP]]`. The row and column labels determine how the counts should be interpreted.

Here a positive prediction means *flagged as a default risk*. It does not by itself prescribe approval or rejection.

In [ ]:
cm = confusion_matrix(y_dev, oof_class, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.8, 4.5))
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=['No default (0)', 'Default (1)']
).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('Out-of-fold predictions at threshold 0.5')
plt.tight_layout()
plt.show()

Read the bottom-left entry: these borrowers defaulted but the model predicted no default. Read the top-right entry: these borrowers did not default but were flagged. Both are mistakes, but they describe different failures.

`.ravel()` flattens the 2×2 array row by row. The next cell unpacks the counts in **scikit-learn**'s order and computes two metrics directly. This dataset contains actual defaults and non-defaults, so both denominators below are positive.

In [ ]:
tn, fp, fn, tp = cm.ravel()
manual_accuracy = (tp + tn) / cm.sum()
manual_recall = tp / (tp + fn)
print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'Accuracy by hand: {manual_accuracy:.3f}')
print(f'Recall by hand:   {manual_recall:.3f}')

The metric functions below calculate the same measures and add specificity and precision. Recall with `pos_label=0` is the fraction of non-defaults correctly classified, so it gives specificity. For precision, `zero_division=0` asks scikit-learn to return zero if a model flags nobody; in that case precision is mathematically undefined, and a zero output reflects the convention rather than an observed precision among flagged cases.

In [ ]:
metric_table = pd.Series({
    'Accuracy': accuracy_score(y_dev, oof_class),
    'Recall (default)': recall_score(y_dev, oof_class, pos_label=1),
    'Specificity (no default)': recall_score(y_dev, oof_class, pos_label=0),
    'Precision (default)': precision_score(y_dev, oof_class, pos_label=1, zero_division=0)
}, name='Out-of-fold metric')
metric_table.round(3)

Interpret each denominator: recall asks how many **actual defaults** were found; precision asks how many **flagged borrowers** actually defaulted. They answer different questions. Accuracy aggregates correct decisions from both classes and can hide weak performance on one class.

### Compare against a simple baseline

A `DummyClassifier(strategy='most_frequent')` learns only the majority class in each training fold. The code evaluates it on the **same folds** as logistic regression. This tests whether the model adds predictive information beyond always choosing the most common outcome.

In [ ]:
baseline_cv = cross_validate(
    DummyClassifier(strategy='most_frequent'), X_dev, y_dev,
    cv=folds, scoring='accuracy'
)
pd.Series({
    'Logistic regression': cv_results['test_accuracy'].mean(),
    'Majority-class baseline': baseline_cv['test_score'].mean()
}, name='Mean validation accuracy').round(3)

### ROC analysis: use scores, not class labels

![Evaluation lecture slide 21: confusion matrices at successive cut-offs trace points in ROC space.](../resources/3_roc_construction.png)

The ROC calculation uses `oof_probability` rather than `oof_class`. Binary class labels would retain only one operating point and discard the ordering supplied by the probabilities.

The diagonal represents random ranking in expectation (AUC 0.5); the upper-left corner represents perfect separation. Empirical curves can cross or fall below the diagonal. AUC 1 means perfect ranking on the evaluated observations, not guaranteed perfection on future data.

The slide uses a strict `>` comparison at exact score ties; our calculations use `>=` throughout, as in the rule in Key Concepts. This convention determines which side receives observations exactly at the threshold.

In [ ]:
oof_auc = roc_auc_score(y_dev, oof_probability)
false_positive_rate = fp / (fp + tn)
true_positive_rate = tp / (tp + fn)

fig, ax = plt.subplots(figsize=(6.2, 5))
RocCurveDisplay.from_predictions(
    y_dev, oof_probability, pos_label=1, name='Out-of-fold logit', ax=ax
)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random ranking')
ax.scatter(false_positive_rate, true_positive_rate, color='black', s=55,
           zorder=3, label='Threshold 0.5')
ax.set(xlabel='False-positive rate (1 − specificity)',
       ylabel='True-positive rate (recall)', title='ROC analysis of default-risk predictions')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print(f'Pooled out-of-fold ROC-AUC: {oof_auc:.3f}')

The black point is the confusion matrix you already examined, expressed as two rates. The ROC curve shows how the operating point varies while the underlying scores stay fixed.

The pooled out-of-fold AUC can differ from the mean fold AUC above: pooling compares scores across different fitted models, whereas a fold AUC compares scores within one model's validation fold. The labels distinguish these two summaries. AUC measures discrimination; it does not establish the numerical accuracy of probabilities or business value. There is no universal AUC threshold that makes a model suitable for deployment.

## Bias, variance and overfitting

![Evaluation lecture slide 26: insufficient flexibility, excessive flexibility and a useful intermediate fit.](../resources/3_overfitting.png)

### A simple way to change flexibility

A **nearest-neighbor classifier** makes the lecture's idea concrete. For a new application, it finds nearby training applications and uses their class votes. `n_neighbors` specifies how many neighbors vote:

- **1 neighbor:** extremely local predictions; a training point can effectively predict its own label.
- **25 neighbors:** averages over a neighborhood rather than copying one observation.
- **500 neighbors:** averages over much of this small sample and can miss local structure.

These are three fixed teaching settings, not a comprehensive model search. We assess the same development folds used above and keep the final test set reserved.

Distances depend on feature units. Bureau score spans hundreds of points, while debt/income spans tens of percentage points. `StandardScaler` expresses each feature relative to its training mean and standard deviation. A **pipeline** applies scaling followed by classification; cross-validation fits the scaler separately inside each training fold. This keeps validation information out of preprocessing.

In [ ]:
neighbor_counts = [1, 25, 500]
complexity_results = []
for neighbors in neighbor_counts:
    candidate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=neighbors))
    result = cross_validate(
        candidate, X_dev, y_dev, cv=folds,
        scoring='accuracy', return_train_score=True
    )
    complexity_results.append({
        'neighbors': neighbors,
        'training_accuracy': result['train_score'].mean(),
        'validation_accuracy': result['test_score'].mean()
    })
complexity_table = pd.DataFrame(complexity_results).set_index('neighbors')
complexity_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
complexity_table.plot.bar(ax=ax, color=['#2166ac', '#e08214'], rot=0)
ax.axhline(baseline_cv['test_score'].mean(), linestyle='--', color='gray',
           label='Majority baseline (validation)')
ax.set(xlabel='Number of neighbors (larger neighborhoods smooth more)',
       ylabel='Mean accuracy across folds', ylim=(0, 1.08),
       title='Training fit and validation performance can disagree')
ax.legend(['Majority baseline (validation)', 'Training', 'Validation'], loc='lower right')
plt.tight_layout()
plt.show()

Compare the 1-neighbor training score with its validation score. A training point is its own nearest neighbor, so near-perfect training accuracy here is unsurprising. The validation gap reveals that copying training labels does not generalize equally well.

With 500 neighbors, the model averages over a large part of the sample. If both scores approach the majority baseline, it is failing to exploit much of the feature information. Inspect whether the intermediate neighborhood improves validation accuracy.

This illustrates flexibility and overfitting. It does **not** empirically decompose error into bias and variance: assessing sensitivity to training samples requires more than these three mean scores. Perfect training accuracy alone is also insufficient to diagnose overfitting; the comparison with unseen cases matters.

### Freeze the choice, then assess the reserved test set

The code chooses the neighbor count with the highest **mean validation accuracy**, rather than training accuracy. `idxmax()` returns the row label of the first maximum. It then fits that pipeline on all development applications. Logistic regression remains our fixed reference model, and the majority classifier remains the baseline.

The three models below form a predeclared final comparison. Their test results provide the final assessment. Further choices based on those results would turn the test set into additional development data.

In [ ]:
best_neighbors = int(complexity_table['validation_accuracy'].idxmax())
selected_knn = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=best_neighbors)
)
print(f'Neighbor count chosen using validation accuracy: {best_neighbors}')

final_models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': LogisticRegression(solver='newton-cholesky'),
    'Selected nearest neighbors': selected_knn
}
final_results = []
for name, model in final_models.items():
    model.fit(X_dev, y_dev)
    test_probability = model.predict_proba(X_test)[:, 1]
    test_class = (test_probability >= cutoff).astype(int)
    final_results.append({
        'model': name,
        'test_accuracy': accuracy_score(y_test, test_class),
        'test_roc_auc': roc_auc_score(y_test, test_probability)
    })
final_table = pd.DataFrame(final_results).set_index('model')
final_table.round(3)

Compare the final results with the earlier validation estimates. Differences are expected: the test observations differ, and final models use more training data than the fold models. A small advantage on this one test set does not establish that an algorithm is universally better.

The lecture also asks about scalability, robustness, comprehensibility, justifiability and fairness. The metrics above address predictive performance on these synthetic applications; they do not assess all those other dimensions.

## Summary

- The metric and the separation of unseen observations define what an evaluation can establish.
- `cross_validate(..., scoring=...)` makes evaluation explicit; its `test_` results refer to validation folds.
- `cross_val_predict` provides row-aligned out-of-fold predictions for detailed inspection.
- Confusion-matrix interpretation depends on the axes and positive class. In **scikit-learn**, rows are actual classes and columns are predicted classes.
- ROC analysis uses probabilities or scores; confusion counts and class-based metrics use class decisions.
- Training–validation comparisons help diagnose problems; validation guides choices, while the final test set provides a separate assessment.
- High training accuracy is not enough. A useful model must capture structure that carries over to new observations.

Here's a quick reference for the new Python tools introduced in this notebook:

| Python pattern | What it helps you do |
|---|---|
| `StratifiedKFold` | Define classification folds with similar class proportions |
| `cross_val_predict(..., method='predict_proba')` | Collect row-aligned out-of-fold probabilities |
| `confusion_matrix`, `ConfusionMatrixDisplay` | Count and visualize correct and incorrect class decisions |
| `accuracy_score`, `recall_score`, `precision_score` | Calculate class-based performance measures |
| `RocCurveDisplay.from_predictions`, `roc_auc_score` | Plot an ROC curve and summarize ranking performance |
| `DummyClassifier` | Establish a majority-class baseline |
| `make_pipeline`, `StandardScaler`, `KNeighborsClassifier` | Combine training-fold scaling with a nearest-neighbor classifier |
| `.idxmax()` | Identify the neighbor count with the highest validation accuracy |
| `.plot.bar`, `ax.axhline` | Compare model results and show a reference line |

## Tasks

Use the working cells below. Keep the reserved test results fixed while practicing; the development predictions and fold results provide everything you need.

### Task 1: Audit the confusion matrix

Use the existing `cm` to calculate specificity and precision directly. Compare your answers with `metric_table`. Explain the denominator of each measure in words, and explain why transposing the matrix would change how you extract its counts.

*Hint:* scikit-learn orders the flattened entries as TN, FP, FN, TP. If a denominator is zero, explicitly report that the ratio is undefined.

In [ ]:
# YOUR CODE: calculate and interpret specificity and precision.

### Task 2: Inspect the errors

Filter `predictions` into false positives and false negatives using its observed and predicted columns. Display three rows from each group, retaining their probabilities and features. Explain how a wrong class prediction can still come from a probability close to 0.5.

*Hint:* combine Boolean conditions with `&` and put each condition in parentheses.

In [ ]:
# YOUR CODE: select examples of both kinds of classification error.

### Task 3: Display fold variation

Plot the five `validation_accuracy` values from `fold_results` and add a horizontal line at their mean. Label the fold axis and the accuracy axis. Describe the variation without treating the five folds as five independent experiments or presenting their spread as a confidence interval.

*Hint:* use `ax.plot` and `ax.axhline`.

In [ ]:
# YOUR CODE: visualize validation accuracy across folds.